# Organización de hábitos a partir de registros de salud
## Pipeline de Machine Learning: desde los datos crudos hasta la tokenización

**Problema:** una persona registra a diario cómo durmió, cuánto caminó y escribe una nota corta sobre su día ("dormí pésimo, mucho estrés con las pruebas 😩"). Queremos que, más adelante, un modelo lea esas notas y ayude a organizar sus hábitos (por ejemplo, detectar que el mal sueño se repite en semanas de pruebas).

Este notebook cubre **todo lo que ocurre antes del modelo**, hasta dejar el texto convertido en números que un modelo puede recibir:

| Etapa | Qué hacemos | Por qué importa |
|---|---|---|
| 1. Datos | Generamos un dataset realista (sintético) | No usamos datos reales de salud: son sensibles |
| 2. Exploración | Revisamos tamaño, vacíos y errores | Saber qué hay que arreglar |
| 3. Limpieza | Duplicados, vacíos, valores imposibles | Un dato malo contamina todo lo que sigue |
| 4. Anonimización | Borramos correos, teléfonos y RUT | Privacidad: los datos de salud son sensibles |
| 5. Normalización | Minúsculas, tildes, emojis, chilenismos | Que "Dormí", "dormi" y "DORMÍ" sean lo mismo |
| 6. Tokenización | Por palabras y por subpalabras (BPE) | Partir el texto en unidades que el modelo entiende |
| 7. IDs y padding | Convertimos tokens en números del mismo largo | Es la entrada real de un modelo |

> El modelo en sí (clasificar notas, recomendar hábitos) queda **fuera** de este notebook: aquí terminamos con los datos listos para entrenarlo.

## 0. Preparación

In [ ]:
!pip -q install tokenizers

import re
import random
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
pd.set_option("display.max_colwidth", 120)

## 1. Datos

Usamos datos **sintéticos**: los generamos con código imitando cómo escribe la gente en una app de hábitos en Chile (chilenismos, emojis, letras repetidas, errores de tildes). Así evitamos usar datos de salud de personas reales.

Cada registro tiene:
- `usuario_id`, `fecha`
- `horas_sueno`, `pasos`, `vasos_agua` → datos numéricos (como los de un reloj inteligente)
- `nota` → texto libre escrito por la persona (**esto es lo que vamos a tokenizar**)
- `categoria` → tema principal de la nota. No la usamos aún: sirve para entrenar el modelo en la siguiente etapa.

A propósito metemos **suciedad**: filas duplicadas, notas vacías, horas de sueño imposibles y datos personales dentro del texto.

In [ ]:
FRASES = {
    "sueno": [
        "dormí pésimo anoche", "me quedé despierto hasta las 3 viendo series",
        "dormí como 5 horas nomás", "tuve insomnio de nuevo",
        "dormí bacán, 8 horas", "me desperté muuuy cansado",
        "me acosté tarde por el celular", "dormí la siesta y en la noche no pude dormir",
    ],
    "estres": [
        "mucho estrés con las pruebas de la u", "harta pega en el trabajo",
        "ando con ansiedad por el certamen", "semana pesada, cero tiempo",
        "me tiene chato la tesis", "demasiadas entregas esta semana",
    ],
    "alimentacion": [
        "almorcé completo con papas fritas", "comí puras sopaipillas",
        "me salté el desayuno otra vez", "comí ensalada y pollo",
        "tomé como 4 cafés", "cené tarde y pesado", "tomé harta agua hoy",
    ],
    "ejercicio": [
        "salí a trotar 30min", "fui al gym po", "caminé harto hoy",
        "cero ejercicio, puro sillón", "jugué pichanga con los cabros",
        "hice yoga en la mañana", "subí escaleras en vez del ascensor",
    ],
    "animo": [
        "me siento bacán hoy", "ando bajoneado", "día tranqui",
        "estoy feliz, salió todo bien", "ando medio triste", "me siento con energía",
    ],
}
EMOJIS = {"sueno": ["😴", "🥱"], "estres": ["😩", "😰"], "alimentacion": ["🍔", "☕", "🥗"],
          "ejercicio": ["💪", "🏃"], "animo": ["😊", "😢"]}
MULETILLAS = [" po", " cachai", "!!!", "...", " jaja"]
DATOS_PERSONALES = [
    " le escribí a la Cata al +56 9 8765 4321",
    " mi correo es juanito.perez@gmail.com",
    " para la licencia me pidieron el RUT 12.345.678-9",
    " llamé al doctor al 987654321",
]

def ensuciar(texto):
    # Imita errores reales de escritura
    if random.random() < 0.30:                       # sin tildes
        texto = "".join(c for c in unicodedata.normalize("NFD", texto)
                        if unicodedata.category(c) != "Mn" or c == "̃")
        texto = unicodedata.normalize("NFC", texto)
    if random.random() < 0.10:
        texto = texto.upper()                        # todo en mayúsculas
    if random.random() < 0.15:                       # letras alargadas: "muuuy"
        texto = re.sub(r"([aeiou])", lambda m: m.group(1) * 3, texto, count=1)
    return texto

def generar_nota():
    cat = random.choice(list(FRASES))
    partes = [random.choice(FRASES[cat])]
    if random.random() < 0.5:                        # una segunda idea de otro tema
        otra = random.choice(list(FRASES))
        partes.append(random.choice(FRASES[otra]))
    nota = ", ".join(partes)
    if random.random() < 0.4:
        nota += random.choice(MULETILLAS)
    if random.random() < 0.5:
        nota += " " + random.choice(EMOJIS[cat])
    if random.random() < 0.06:
        nota += random.choice(DATOS_PERSONALES)
    return ensuciar(nota), cat

filas = []
fechas = pd.date_range("2026-08-01", periods=30, freq="D")
for u in range(1, 21):
    for f in random.sample(list(fechas), k=random.randint(12, 25)):
        nota, cat = generar_nota()
        filas.append({
            "usuario_id": f"u{u:02d}",
            "fecha": f.date(),
            "horas_sueno": round(float(np.clip(np.random.normal(6.5, 1.4), 2, 11)), 1),
            "pasos": int(max(0, np.random.normal(6500, 2800))),
            "vasos_agua": int(np.random.randint(0, 11)),
            "nota": nota,
            "categoria": cat,
        })

df_crudo = pd.DataFrame(filas)

# Suciedad intencional
df_crudo.loc[df_crudo.sample(6, random_state=1).index, "horas_sueno"] = [25, -1, 30, 0, 48, -3]
df_crudo.loc[df_crudo.sample(8, random_state=2).index, "nota"] = None
df_crudo.loc[df_crudo.sample(5, random_state=3).index, "nota"] = "   "
df_crudo.loc[df_crudo.sample(4, random_state=4).index, "nota"] = "😴"
df_crudo = pd.concat([df_crudo, df_crudo.sample(10, random_state=5)], ignore_index=True)  # duplicados

print("Filas:", len(df_crudo))
df_crudo.sample(8, random_state=7)

## 2. Exploración: ¿qué tan sucios están los datos?

In [ ]:
print("Filas duplicadas:     ", df_crudo.duplicated().sum())
print("Notas nulas:          ", df_crudo["nota"].isna().sum())
print("Notas solo espacios:  ", (df_crudo["nota"].fillna("").str.strip() == "").sum() - df_crudo["nota"].isna().sum())
print("Horas de sueño fuera de 0–16:", ((df_crudo["horas_sueno"] <= 0) | (df_crudo["horas_sueno"] > 16)).sum())
print("Notas con correo:     ", df_crudo["nota"].fillna("").str.contains("@").sum())
print("Notas con teléfono:   ", df_crudo["nota"].fillna("").str.contains(r"\d{3}\s?\d{4}").sum())
print()
print(df_crudo["categoria"].value_counts())

## 3. Limpieza

Reglas que aplicamos (y por qué):

1. **Duplicados** → se borran. Si una nota aparece dos veces, el modelo la "aprende el doble".
2. **Notas vacías o solo espacios** → se borran. No hay texto que tokenizar.
3. **Horas de sueño imposibles** (≤ 0 o > 16) → se dejan como vacías (`NaN`) en vez de borrar la fila, porque la nota sigue siendo útil.

In [ ]:
df = df_crudo.copy()
n0 = len(df)

df = df.drop_duplicates()
n1 = len(df)

df["nota"] = df["nota"].fillna("").str.strip()
df = df[df["nota"] != ""]
n2 = len(df)

invalidas = (df["horas_sueno"] <= 0) | (df["horas_sueno"] > 16)
df.loc[invalidas, "horas_sueno"] = np.nan

df = df.reset_index(drop=True)
print(f"Inicio: {n0} | sin duplicados: {n1} | sin notas vacías: {n2}")
print(f"Horas de sueño marcadas como inválidas: {invalidas.sum()}")

## 4. Anonimización

Algunas notas traen datos personales: correos, teléfonos y RUT. Los reemplazamos por **etiquetas** (`<email>`, `<telefono>`, `<rut>`). Así el modelo sabe que "ahí había un teléfono" sin conocer el número.

Esto se hace **antes** de normalizar, porque después de quitar puntos y guiones ya no se reconoce un RUT como `12.345.678-9`.

> **Limitación honesta:** los nombres propios ("la Cata") no se detectan con reglas simples. Para eso se necesita un modelo de reconocimiento de entidades (NER), que queda fuera de este alcance.

In [ ]:
PATRONES = [
    (r"[\w.+-]+@[\w-]+\.[\w.]+",                          "<email>"),
    (r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b",               "<rut>"),
    (r"(\+?56\s?)?9\s?\d{4}\s?\d{4}\b",                   "<telefono>"),
]

def anonimizar(texto):
    for patron, etiqueta in PATRONES:
        texto = re.sub(patron, etiqueta, texto)
    return texto

df["nota_anon"] = df["nota"].apply(anonimizar)

cambiadas = df[df["nota"] != df["nota_anon"]]
print("Notas anonimizadas:", len(cambiadas))
cambiadas[["nota", "nota_anon"]].head()

## 5. Normalización

Buscamos que textos que **significan lo mismo** queden **escritos igual**. Cada paso es una decisión de diseño:

| Paso | Ejemplo | Decisión |
|---|---|---|
| Minúsculas | `DORMÍ` → `dormí` | Las mayúsculas no cambian el sentido aquí |
| Emojis → etiquetas | `😴` → `<emo_sueno>` | El emoji **sí** aporta información: no se borra |
| Letras alargadas | `muuuy` → `muy` | 3 o más letras iguales seguidas se reducen a una |
| Quitar tildes (menos la ñ) | `dormí` → `dormi` | La gente escribe con y sin tildes; así se unifican. La ñ se conserva (`año` ≠ `ano`) |
| Separar número y unidad | `30min` → `30 min` | Para que `30` y `min` sean tokens distintos |
| Chilenismos | `pega` → `trabajo`, `bacan` → `bueno` | Reduce vocabulario. **Costo:** se pierde algo de matiz |
| Puntuación | `!!!`, `...` → se borra | Se conservan `<` `>` de las etiquetas |

In [ ]:
EMOJI_A_TOKEN = {
    "😴": "<emo_sueno>", "🥱": "<emo_sueno>", "😩": "<emo_estres>", "😰": "<emo_estres>",
    "🍔": "<emo_comida>", "🥗": "<emo_comida>", "☕": "<emo_cafe>",
    "💪": "<emo_ejercicio>", "🏃": "<emo_ejercicio>", "😊": "<emo_feliz>", "😢": "<emo_triste>",
}

# Diccionario de chilenismos (se aplica ya sin tildes)
CHILENISMOS = {
    "po": "", "pos": "", "cachai": "", "jaja": "",
    "harto": "mucho", "harta": "mucha", "bacan": "bueno", "tranqui": "tranquilo",
    "pega": "trabajo", "chato": "cansado", "bajoneado": "triste",
    "nomas": "solamente", "u": "universidad", "cabros": "amigos",
    "pichanga": "futbol", "certamen": "prueba", "gym": "gimnasio",
}

def quitar_tildes(texto):
    # Quita tildes pero conserva la ñ
    return "".join(c if c == "ñ" else unicodedata.normalize("NFD", c)[0] for c in texto)

def normalizar(texto):
    texto = texto.lower()
    for emoji, token in EMOJI_A_TOKEN.items():
        texto = texto.replace(emoji, f" {token} ")
    texto = re.sub(r"(\w)\1{2,}", r"\1", texto)            # muuuy -> muy
    texto = quitar_tildes(texto)
    texto = re.sub(r"(\d)([a-zñ])", r"\1 \2", texto)       # 30min -> 30 min
    texto = re.sub(r"[^\w\sñ<>]", " ", texto)               # puntuación fuera
    palabras = [CHILENISMOS.get(p, p) for p in texto.split()]
    return " ".join(p for p in palabras if p)

df["nota_norm"] = df["nota_anon"].apply(normalizar)
df = df[df["nota_norm"] != ""].reset_index(drop=True)

df[["nota", "nota_norm"]].sample(10, random_state=3)

### División entrenamiento / prueba

Separamos **antes** de construir el vocabulario. El vocabulario se aprende solo con el 80% de entrenamiento; el 20% de prueba simula notas nuevas que el sistema nunca vio. Así podemos medir cuántas palabras **desconocidas** aparecen en la vida real.

In [ ]:
df = df.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
corte = int(len(df) * 0.8)
train, test = df.iloc[:corte].copy(), df.iloc[corte:].copy()
print("Entrenamiento:", len(train), "| Prueba:", len(test))

## 6A. Tokenización por palabras

La forma más simple: cada palabra es un token. Las etiquetas como `<emo_sueno>` se mantienen como **un solo token**.

### Palabras vacías (stopwords)
Palabras como "de", "la", "el" aparecen en casi todas las notas y aportan poco. Las quitamos, **pero con cuidado**: las listas típicas de stopwords incluyen "no", y aquí sería un error grave:

- "dormí bien" vs. "**no** dormí bien" → significan lo contrario.

Por eso usamos una lista propia que **conserva las negaciones** (`no`, `nunca`, `cero`, `nada`).

In [ ]:
STOPWORDS = set('''
a al algo ante con de del desde durante e el ella en entre era es esta este esto
hasta la las le les lo los me mi mis para por que se si su sus te tu un una uno unos
y ya yo como mas muy otra otro vez hoy
'''.split())
NEGACIONES = {"no", "nunca", "cero", "nada", "sin"}
assert not (STOPWORDS & NEGACIONES), "¡Una negación quedó como stopword!"

PATRON_TOKEN = re.compile(r"<\w+>|\w+")

def tokenizar_palabras(texto, quitar_stopwords=True):
    tokens = PATRON_TOKEN.findall(texto)
    if quitar_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

train["tokens"] = train["nota_norm"].apply(tokenizar_palabras)
test["tokens"] = test["nota_norm"].apply(tokenizar_palabras)

for _, fila in train.head(5).iterrows():
    print(f"{fila['nota']!r}\n   -> {fila['tokens']}\n")

### Vocabulario y conversión a números

Un modelo no entiende palabras, entiende números. Construimos un **vocabulario**: a cada token le damos un número (ID).

- Solo entran palabras que aparecen **al menos 2 veces** (las demás suelen ser errores de tipeo).
- Dos tokens especiales: `<pad>` = 0 (relleno) y `<unk>` = 1 (palabra desconocida).

In [ ]:
FRECUENCIA_MINIMA = 2
conteo = Counter(t for tokens in train["tokens"] for t in tokens)

vocab = {"<pad>": 0, "<unk>": 1}
for token, frec in conteo.most_common():
    if frec >= FRECUENCIA_MINIMA:
        vocab[token] = len(vocab)
id_a_token = {i: t for t, i in vocab.items()}

print("Tokens distintos en entrenamiento:", len(conteo))
print("Tamaño del vocabulario (frec ≥ 2): ", len(vocab))
print("Primeros 20:", list(vocab.items())[:20])

In [ ]:
top = conteo.most_common(15)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh([t for t, _ in top][::-1], [c for _, c in top][::-1], color="#3b82f6")
ax.set_title("15 tokens más frecuentes (entrenamiento)")
ax.set_xlabel("Apariciones")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 6B. Tokenización por subpalabras (BPE)

La tokenización por palabras tiene un problema: si llega una palabra que no está en el vocabulario ("trasnoché"), se pierde completa como `<unk>`.

**BPE (Byte Pair Encoding)** es lo que usan los modelos de lenguaje modernos. Parte de letras sueltas y va **uniendo los pares que más se repiten** hasta formar pedazos útiles: `dorm`, `i`, `cans`, `ado`. Así, una palabra nueva se puede armar con pedazos conocidos.

Entrenamos nuestro propio BPE solo con las notas de entrenamiento, usando la librería `tokenizers` de Hugging Face.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

ESPECIALES = ["<pad>", "<unk>", "<email>", "<telefono>", "<rut>"] + sorted(set(EMOJI_A_TOKEN.values()))

bpe = Tokenizer(models.BPE(unk_token="<unk>"))
bpe.pre_tokenizer = pre_tokenizers.Whitespace()
entrenador = trainers.BpeTrainer(vocab_size=400, min_frequency=2, special_tokens=ESPECIALES)
bpe.train_from_iterator(train["nota_norm"].tolist(), trainer=entrenador)

print("Tamaño del vocabulario BPE:", bpe.get_vocab_size())

In [ ]:
ejemplos = [
    "dormi pesimo anoche <emo_sueno>",
    "trasnoche estudiando y amaneci agotado",      # palabras que NO están en los datos
    "no dormi bien por la prueba de calculo",
]
for texto in ejemplos:
    por_palabra = [t if t in vocab else "<unk>" for t in tokenizar_palabras(texto, quitar_stopwords=False)]
    print("Texto:     ", texto)
    print("Palabras:  ", por_palabra)
    print("BPE:       ", bpe.encode(texto).tokens)
    print()

Fíjate en la segunda oración: por palabras casi todo queda como `<unk>`, mientras que BPE la reconstruye con pedazos que sí conoce. Esa es la ventaja principal de las subpalabras.

## 7. IDs, relleno (padding) y recorte

Los modelos reciben lotes de **tamaño fijo**: todas las notas deben tener el mismo número de tokens.

- Notas cortas → se **rellenan** con `<pad>` (ID 0) al final.
- Notas largas → se **recortan**.

Elegimos el largo máximo con el **percentil 95** del entrenamiento: cubre al 95% de las notas sin desperdiciar espacio en las muy largas.

In [ ]:
largos = train["tokens"].apply(len)
MAX_LEN = int(np.percentile(largos, 95))
print("Largo máximo elegido (percentil 95):", MAX_LEN)

def a_ids(tokens, max_len=MAX_LEN):
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens][:max_len]
    return ids + [vocab["<pad>"]] * (max_len - len(ids))

X_train_palabras = np.array(train["tokens"].apply(a_ids).tolist())
X_test_palabras = np.array(test["tokens"].apply(a_ids).tolist())

# Lo mismo con BPE
largos_bpe = [len(e.tokens) for e in bpe.encode_batch(train["nota_norm"].tolist())]
MAX_LEN_BPE = int(np.percentile(largos_bpe, 95))
bpe.enable_padding(pad_id=bpe.token_to_id("<pad>"), pad_token="<pad>", length=MAX_LEN_BPE)
bpe.enable_truncation(max_length=MAX_LEN_BPE)
X_train_bpe = np.array([e.ids for e in bpe.encode_batch(train["nota_norm"].tolist())])
X_test_bpe = np.array([e.ids for e in bpe.encode_batch(test["nota_norm"].tolist())])

print("Matriz palabras (entrenamiento):", X_train_palabras.shape)
print("Matriz BPE (entrenamiento):     ", X_train_bpe.shape)
print()
i = 0
print("Nota:  ", train.iloc[i]["nota"])
print("Tokens:", train.iloc[i]["tokens"])
print("IDs:   ", X_train_palabras[i].tolist())
print("Vuelta:", [id_a_token[j] for j in X_train_palabras[i] if j != 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(largos, bins=range(0, max(largos) + 2), color="#3b82f6", edgecolor="white")
axes[0].axvline(MAX_LEN, color="#ef4444", linestyle="--", label=f"corte = {MAX_LEN}")
axes[0].set_title("Tokens por nota — por palabras")
axes[1].hist(largos_bpe, bins=range(0, max(largos_bpe) + 2), color="#10b981", edgecolor="white")
axes[1].axvline(MAX_LEN_BPE, color="#ef4444", linestyle="--", label=f"corte = {MAX_LEN_BPE}")
axes[1].set_title("Tokens por nota — BPE")
for ax in axes:
    ax.set_xlabel("Tokens"); ax.set_ylabel("Notas"); ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 8. Comparación: palabras vs. BPE

La métrica clave es la **tasa de desconocidos (OOV)** en el conjunto de prueba: qué porcentaje de tokens el sistema no reconoce en notas nuevas. Una tasa alta significa que el modelo va a "no entender" parte de lo que escriben los usuarios.

In [ ]:
def tasa_oov(matriz, unk_id, pad_id=0):
    reales = matriz[matriz != pad_id]
    return 100 * (reales == unk_id).mean()

resumen = pd.DataFrame({
    "Palabras": [len(vocab), np.mean(largos), tasa_oov(X_test_palabras, vocab["<unk>"])],
    "BPE (subpalabras)": [bpe.get_vocab_size(), np.mean(largos_bpe), tasa_oov(X_test_bpe, bpe.token_to_id("<unk>"))],
}, index=["Tamaño del vocabulario", "Tokens promedio por nota", "% desconocidos en prueba (OOV)"]).round(2)
resumen

**Cómo leer el resultado:**
- **Palabras:** tokens más cortos por nota y fáciles de interpretar, pero con palabras desconocidas en notas nuevas.
- **BPE:** más tokens por nota (parte las palabras), pero casi sin desconocidos. Es lo que usan los modelos modernos.

Como los datos son sintéticos y repetitivos, el OOV por palabras sale bajo. **Con notas reales de usuarios**, que usan mucho más vocabulario y errores de tipeo, la diferencia a favor de BPE sería bastante mayor.

## 9. (Opcional) Comparar con un tokenizador ya entrenado: BETO

BETO es un modelo BERT entrenado en español por la Universidad de Chile. Su tokenizador aprendió de millones de textos, así que podemos compararlo con el nuestro. Requiere internet (en Colab funciona directo).

In [ ]:
try:
    from transformers import AutoTokenizer
    beto = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")
    for texto in ["me tiene chato la tesis, dormi como 5 horas nomas",
                  "harta pega, cero ejercicio po"]:
        print("Texto:", texto)
        print("BETO: ", beto.tokenize(texto))
        print("Nuestro BPE:", [t for t in bpe.encode(normalizar(texto)).tokens if t != "<pad>"])
        print()
except Exception as e:
    print("No se pudo cargar BETO (¿sin internet?):", type(e).__name__)

## 10. Guardar resultados

Dejamos guardados los datos limpios, el tokenizador BPE y las matrices de IDs, listos para la siguiente etapa (entrenar el modelo).

In [ ]:
import json
train.drop(columns=["tokens"]).to_csv("habitos_train_limpio.csv", index=False)
test.drop(columns=["tokens"]).to_csv("habitos_test_limpio.csv", index=False)
bpe.save("tokenizador_bpe_habitos.json")
with open("vocab_palabras.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=1)
np.savez("habitos_ids.npz", X_train_palabras=X_train_palabras, X_test_palabras=X_test_palabras,
         X_train_bpe=X_train_bpe, X_test_bpe=X_test_bpe,
         y_train=train["categoria"].values, y_test=test["categoria"].values)
print("Archivos guardados.")

## 11. Pruebas del pipeline

Todavía no hay modelo, así que lo que probamos es que **cada etapa haga lo que promete**. Usamos `assert`: si una regla se rompe, la celda se detiene y muestra cuál falló. Si no aparece ningún error, todas pasaron.

Hay tres tipos de prueba:
1. **Casos conocidos:** le damos un texto y comparamos con el resultado esperado.
2. **Propiedades sobre todo el dataset:** reglas que deben cumplirse en *todas* las filas (por ejemplo, que no quede ningún correo).
3. **Casos borde:** texto vacío, solo un emoji o una nota muy larga.

In [ ]:
pruebas_ok = 0
def prueba(nombre, condicion):
    global pruebas_ok
    assert condicion, f"FALLÓ: {nombre}"
    pruebas_ok += 1
    print(f"✔ {nombre}")

# --- 1. Casos conocidos ---------------------------------------------------
prueba("Minúsculas, letras alargadas, tildes, chilenismo y emoji",
       normalizar("MUUUY bacán el día 😴") == "muy bueno el dia <emo_sueno>")
prueba("La ñ se conserva",
       normalizar("Año") == "año")
prueba("Separa número y unidad",
       normalizar("troté 30min") == "trote 30 min")
prueba("La negación 'no' sobrevive a las stopwords",
       "no" in tokenizar_palabras(normalizar("No dormí bien")))
prueba("'dormí bien' y 'no dormí bien' quedan distintos",
       tokenizar_palabras(normalizar("dormí bien")) != tokenizar_palabras(normalizar("no dormí bien")))
prueba("Escrituras distintas de la misma palabra quedan iguales",
       normalizar("Dormí") == normalizar("dormi") == normalizar("DORMÍ"))
prueba("Anonimiza correo",  anonimizar("mi correo es ana.soto@uc.cl") == "mi correo es <email>")
prueba("Anonimiza teléfono", anonimizar("llámame al +56 9 1234 5678") == "llámame al <telefono>")
prueba("Anonimiza RUT",      anonimizar("RUT 9.876.543-K") == "RUT <rut>")
prueba("Las etiquetas son un solo token",
       tokenizar_palabras("<emo_sueno> <email>") == ["<emo_sueno>", "<email>"])
prueba("BPE no parte las etiquetas",
       "<telefono>" in bpe.encode("llame al <telefono>").tokens)

# --- 2. Propiedades sobre todo el dataset ---------------------------------
prueba("No queda ningún correo en el texto final",
       not df["nota_norm"].str.contains("@").any())
prueba("No queda ningún número de 8 o más dígitos (teléfonos/RUT)",
       not df["nota_norm"].str.replace(" ", "").str.contains(r"\d{8,}").any())
prueba("No quedan emojis sin convertir",
       not df["nota_norm"].apply(lambda t: any(ord(c) > 0x2600 for c in t)).any())
prueba("No quedan mayúsculas",
       (df["nota_norm"] == df["nota_norm"].str.lower()).all())
prueba("Todas las filas tienen el mismo largo (padding)",
       X_train_palabras.shape[1] == X_test_palabras.shape[1] == MAX_LEN)
prueba("El relleno (0) solo aparece al final de cada fila",
       all(list(f) == sorted(f, key=lambda x: x == 0) for f in X_train_palabras))
# Ida y vuelta: las palabras raras (frecuencia < 2) deben volver como <unk>
def esperado(tokens):
    return [t if t in vocab else "<unk>" for t in tokens][:MAX_LEN]
prueba("IDs → tokens devuelve los mismos tokens (ida y vuelta, raras como <unk>)",
       all([id_a_token[j] for j in X_train_palabras[i] if j != 0] == esperado(train.iloc[i]["tokens"])
           for i in range(len(train))))
prueba("Entrenamiento y prueba no comparten filas",
       len(set(train.index) & set(test.index)) == 0 and len(train) + len(test) == len(df))

# --- 3. Casos borde --------------------------------------------------------
prueba("Texto vacío no rompe nada",           a_ids(tokenizar_palabras(normalizar(""))) == [0] * MAX_LEN)
prueba("Solo un emoji produce un token",      tokenizar_palabras(normalizar("😢")) == ["<emo_triste>"])
prueba("Nota muy larga se recorta",           len(a_ids(tokenizar_palabras(normalizar("dormí mal " * 50)))) == MAX_LEN)
prueba("Palabra nunca vista se vuelve <unk>", a_ids(["xyzzy"])[0] == vocab["<unk>"])

print(f"\n{pruebas_ok} pruebas pasaron ✅")

### Probar con tus propias notas

Escribe cualquier nota y mira qué pasa en cada etapa. Así se ve si el pipeline maneja bien textos que **no están** en el dataset.

In [ ]:
def probar(texto):
    anon = anonimizar(texto)
    norm = normalizar(anon)
    tokens = tokenizar_palabras(norm)
    ids = a_ids(tokens)
    desconocidos = [t for t in tokens if t not in vocab]
    print("Original:      ", texto)
    print("Anonimizado:   ", anon)
    print("Normalizado:   ", norm)
    print("Tokens:        ", tokens)
    print("IDs:           ", ids)
    print("Desconocidos:  ", desconocidos or "ninguno")
    print("BPE:           ", [t for t in bpe.encode(norm).tokens if t != "<pad>"])
    print("-" * 70)

probar("Dormí súper mal, harta pega y cero ejercicio 😩")
probar("Hoy salí a correr 5km y me siento la raja!!! 💪")
probar("me escribió el profe a profe.rojas@umayor.cl por la prueba, ando chato")

### Cómo se testeará el modelo (siguiente etapa)

Cuando exista un modelo, la prueba cambia: ya no se verifican reglas, sino **qué tan bien predice notas que nunca vio**. Para eso dejamos separado el 20% de prueba desde el principio. Las métricas serían:
- **Exactitud (accuracy):** porcentaje de notas bien clasificadas.
- **F1 por categoría:** muestra si el modelo es bueno en todas las categorías o solo en las más frecuentes.
- **Matriz de confusión:** qué categorías confunde entre sí (por ejemplo, sueño con estrés).

**Regla de oro:** el conjunto de prueba se usa **una sola vez, al final**. Si se usa para ajustar el modelo, el resultado deja de ser confiable.

## Conclusiones

1. El texto libre de un registro de hábitos es **muy desordenado**: chilenismos, emojis, letras alargadas, errores de tildes y datos personales. Sin limpieza y normalización, "Dormí", "dormi" y "DORMÍ" serían tres palabras distintas.
2. La **anonimización** es obligatoria con datos de salud y debe hacerse antes de normalizar. Las reglas simples detectan correos, teléfonos y RUT, pero **no nombres**; eso requiere un modelo NER.
3. Al quitar stopwords hay que **conservar las negaciones**: "no dormí bien" y "dormí bien" deben seguir siendo distintas.
4. La tokenización por **palabras** es simple e interpretable. La de **subpalabras (BPE)** maneja mucho mejor las palabras nuevas, a costa de secuencias más largas.
5. El resultado final son **matrices de números de tamaño fijo**, que es exactamente lo que recibe un modelo.

### Siguiente etapa (fuera de este alcance)
- Convertir los IDs en **embeddings** (vectores con significado).
- Entrenar un clasificador de `categoria` y combinar el texto con los datos numéricos (sueño, pasos, agua) para detectar patrones y sugerir hábitos.

---
# Parte 2 (extra): entrenar el modelo una y otra vez

> Esta parte va más allá de la tokenización. Si la tarea solo pide llegar hasta tokenizar, puedes borrar desde aquí hacia abajo.

**Tarea del modelo:** leer una nota y predecir su `categoria` (sueño, estrés, alimentación, ejercicio o ánimo).

"Entrenar una y otra vez" significa tres cosas distintas, y las hacemos las tres:

| Nivel | Qué se repite | Para qué |
|---|---|---|
| **Épocas** | El modelo ve los mismos datos muchas veces | Así es como aprende: ajusta sus pesos en cada pasada |
| **Repeticiones (validación cruzada)** | Entrenamos desde cero con distintas divisiones de los datos | Saber si el resultado es estable o fue suerte |
| **Reentrenamiento** | Agregamos notas nuevas y volvemos a entrenar | Así mejora el sistema en la vida real |

### Regla de oro
Separamos los datos en tres partes:
- **Entrenamiento:** el modelo aprende de aquí.
- **Validación:** la usamos para tomar decisiones (cuántas épocas, qué tamaño de modelo).
- **Prueba:** se usa **una sola vez, al final**. Ya la separamos en la Parte 1 y no la tocamos hasta la sección 16.

## 12. Preparar etiquetas y la división de validación

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

CLASES = sorted(train["categoria"].unique())
clase_a_id = {c: i for i, c in enumerate(CLASES)}

X_todo = X_train_palabras                                  # todo lo que NO es prueba
y_todo = train["categoria"].map(clase_a_id).values
textos_todo = train["nota_norm"].values

X_te = X_test_palabras                                      # prueba: no se toca hasta el final
y_te = test["categoria"].map(clase_a_id).values

idx_tr, idx_val = train_test_split(np.arange(len(X_todo)), test_size=0.2,
                                   stratify=y_todo, random_state=SEMILLA)
print("Clases:", CLASES)
print(f"Entrenamiento: {len(idx_tr)} | Validación: {len(idx_val)} | Prueba (guardada): {len(X_te)}")

## 13. Modelo base: TF-IDF + regresión logística

Antes de una red neuronal, siempre conviene un **modelo simple** como punto de comparación. Si la red neuronal no le gana a esto, no vale la pena.

- **TF-IDF:** representa cada nota según qué palabras tiene, dando más peso a las palabras poco comunes (que suelen ser más informativas).
- **Regresión logística:** aprende qué palabras empujan hacia cada categoría.

In [ ]:
def entrenar_base(textos_tr, y_tr):
    vectorizador = TfidfVectorizer(token_pattern=r"<\w+>|\w+")
    clasificador = LogisticRegression(max_iter=1000)
    clasificador.fit(vectorizador.fit_transform(textos_tr), y_tr)
    return vectorizador, clasificador

vec_base, clf_base = entrenar_base(textos_todo[idx_tr], y_todo[idx_tr])
pred_val = clf_base.predict(vec_base.transform(textos_todo[idx_val]))
print(f"Modelo base — validación: exactitud {accuracy_score(y_todo[idx_val], pred_val):.3f} | "
      f"F1 macro {f1_score(y_todo[idx_val], pred_val, average='macro'):.3f}")

## 14. Red neuronal y ciclo de entrenamiento (épocas)

El modelo es pequeño a propósito:
1. **Embedding:** convierte cada ID en un vector de números que se aprende durante el entrenamiento.
2. **Promedio:** junta los vectores de todos los tokens de la nota (ignorando el relleno `<pad>`).
3. **Capa de salida:** convierte ese promedio en un puntaje por categoría.

En cada **época**, el modelo recorre todos los datos en grupos pequeños (*batches*), compara su predicción con la respuesta correcta (**pérdida**) y ajusta sus pesos para equivocarse menos.

**Parada temprana (early stopping):** si la pérdida en validación deja de mejorar durante varias épocas, paramos y nos quedamos con la mejor versión. Esto evita el **sobreajuste**, que es cuando el modelo memoriza el entrenamiento en vez de aprender.

In [ ]:
class ModeloNotas(nn.Module):
    def __init__(self, tam_vocab, n_clases, dim=32, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(tam_vocab, dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.salida = nn.Linear(dim, n_clases)

    def forward(self, x):
        vectores = self.embedding(x)                          # (notas, tokens, dim)
        mascara = (x != 0).unsqueeze(-1).float()              # 1 = token real, 0 = relleno
        promedio = (vectores * mascara).sum(1) / mascara.sum(1).clamp(min=1)
        return self.salida(self.dropout(promedio))


def entrenar(X_tr, y_tr, X_val=None, y_val=None, epocas=60, lr=0.01, dim=32,
             tam_batch=32, paciencia=6, semilla=0, mostrar=False):
    # Entrena un modelo NUEVO desde cero. Devuelve el modelo y el historial por época.
    torch.manual_seed(semilla)
    rng = np.random.default_rng(semilla)
    modelo = ModeloNotas(len(vocab), len(CLASES), dim=dim)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    perdida_fn = nn.CrossEntropyLoss()

    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    if X_val is not None:
        Xv, yv = torch.tensor(X_val), torch.tensor(y_val)

    historial = {"perdida_train": [], "perdida_val": [], "exactitud_val": []}
    mejor_perdida, mejor_estado, mejor_epoca, sin_mejora = float("inf"), None, 0, 0

    for epoca in range(1, epocas + 1):
        # --- Entrenamiento: una pasada por todos los datos ---
        modelo.train()
        orden = rng.permutation(len(Xt))                      # barajar en cada época
        total = 0.0
        for i in range(0, len(orden), tam_batch):
            lote = orden[i:i + tam_batch]
            optimizador.zero_grad()
            perdida = perdida_fn(modelo(Xt[lote]), yt[lote])
            perdida.backward()                                # calcular cómo ajustar los pesos
            optimizador.step()                                # ajustarlos
            total += perdida.item() * len(lote)
        historial["perdida_train"].append(total / len(Xt))

        # --- Validación: medir sin aprender ---
        if X_val is None:
            continue
        modelo.eval()
        with torch.no_grad():
            salida = modelo(Xv)
            p_val = perdida_fn(salida, yv).item()
            acc_val = (salida.argmax(1) == yv).float().mean().item()
        historial["perdida_val"].append(p_val)
        historial["exactitud_val"].append(acc_val)
        if mostrar and (epoca == 1 or epoca % 5 == 0):
            print(f"Época {epoca:3d} | pérdida train {historial['perdida_train'][-1]:.3f} | "
                  f"pérdida val {p_val:.3f} | exactitud val {acc_val:.3f}")

        # --- Parada temprana ---
        if p_val < mejor_perdida - 1e-4:
            mejor_perdida, mejor_epoca, sin_mejora = p_val, epoca, 0
            mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia:
                if mostrar:
                    print(f"Parada temprana en la época {epoca}. Mejor época: {mejor_epoca}")
                break

    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
    historial["mejor_epoca"] = mejor_epoca if X_val is not None else epocas
    return modelo, historial


def evaluar(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        pred = modelo(torch.tensor(X)).argmax(1).numpy()
    return accuracy_score(y, pred), f1_score(y, pred, average="macro"), pred

In [ ]:
modelo, hist = entrenar(X_todo[idx_tr], y_todo[idx_tr], X_todo[idx_val], y_todo[idx_val], mostrar=True)
acc, f1, _ = evaluar(modelo, X_todo[idx_val], y_todo[idx_val])
print(f"\nRed neuronal — validación: exactitud {acc:.3f} | F1 macro {f1:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist["perdida_train"], label="entrenamiento", color="#3b82f6")
axes[0].plot(hist["perdida_val"], label="validación", color="#f59e0b")
axes[0].axvline(hist["mejor_epoca"] - 1, color="#6b7280", linestyle="--", label="mejor época")
axes[0].set_title("Pérdida por época (más bajo = mejor)")
axes[1].plot(hist["exactitud_val"], color="#10b981")
axes[1].set_title("Exactitud en validación por época")
for ax in axes:
    ax.set_xlabel("Época"); ax.spines[["top", "right"]].set_visible(False)
axes[0].legend()
plt.tight_layout(); plt.show()

**Cómo leer las curvas:** al principio ambas pérdidas bajan, porque el modelo está aprendiendo. Si la de entrenamiento sigue bajando pero la de validación se estanca o sube, el modelo empezó a **memorizar**. Ahí actúa la parada temprana.

## 15. Entrenar una y otra vez: validación cruzada

Un solo entrenamiento puede salir bien o mal **por suerte**: depende de qué notas cayeron en validación y de los pesos iniciales al azar. Para saber cuánto confiar en el resultado, entrenamos **desde cero muchas veces**:

- **Validación cruzada de 5 pliegues:** dividimos los datos en 5 partes. Cada vez usamos 4 para entrenar y 1 para validar, rotando, de modo que todas las notas se usan una vez para validar.
- **Repetida 2 veces**, con otro barajado → **10 entrenamientos por configuración**.

Lo hacemos para tres tamaños de embedding (`dim`) y para el modelo base. Así elegimos la mejor configuración con datos, no a ojo.

In [ ]:
validacion_cruzada = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEMILLA)
pliegues = list(validacion_cruzada.split(X_todo, y_todo))

resultados = []
for dim in [8, 32, 64]:
    for k, (tr, va) in enumerate(pliegues):
        m, h = entrenar(X_todo[tr], y_todo[tr], X_todo[va], y_todo[va], dim=dim, semilla=k)
        acc, f1, _ = evaluar(m, X_todo[va], y_todo[va])
        resultados.append({"modelo": f"red dim={dim}", "pliegue": k, "exactitud": acc,
                           "f1_macro": f1, "mejor_epoca": h["mejor_epoca"]})
    print(f"dim={dim}: listo ({len(pliegues)} entrenamientos)")

for k, (tr, va) in enumerate(pliegues):
    v, c = entrenar_base(textos_todo[tr], y_todo[tr])
    pred = c.predict(v.transform(textos_todo[va]))
    resultados.append({"modelo": "base TF-IDF", "pliegue": k, "exactitud": accuracy_score(y_todo[va], pred),
                       "f1_macro": f1_score(y_todo[va], pred, average="macro"), "mejor_epoca": np.nan})

res = pd.DataFrame(resultados)
resumen_cv = (res.groupby("modelo")[["exactitud", "f1_macro", "mejor_epoca"]]
                 .agg(["mean", "std"]).round(3))
resumen_cv

In [ ]:
orden_modelos = ["base TF-IDF", "red dim=8", "red dim=32", "red dim=64"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([res.loc[res["modelo"] == m, "f1_macro"] for m in orden_modelos])
ax.set_xticks(range(1, len(orden_modelos) + 1), orden_modelos)
for i, m in enumerate(orden_modelos, start=1):
    valores = res.loc[res["modelo"] == m, "f1_macro"]
    ax.scatter(np.full(len(valores), i) + np.random.uniform(-0.08, 0.08, len(valores)),
               valores, color="#3b82f6", alpha=0.6, s=18)
ax.set_title("F1 macro en 10 entrenamientos por modelo")
ax.set_ylabel("F1 macro (validación)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

redes = resumen_cv.loc[[m for m in resumen_cv.index if m.startswith("red")]]
mejor = redes[("f1_macro", "mean")].idxmax()
MEJOR_DIM = int(mejor.split("=")[1])
EPOCAS_FINALES = int(round(res.loc[res["modelo"] == mejor, "mejor_epoca"].median()))
print(f"Mejor red: {mejor} | épocas a usar en el entrenamiento final: {EPOCAS_FINALES}")

**Cómo leer el gráfico:** cada punto es un entrenamiento completo. Si los puntos de un modelo están **muy dispersos**, su resultado depende mucho de la suerte. La **desviación estándar** (`std`) de la tabla mide justamente eso. Es mejor un modelo un poco menos preciso pero estable que uno que a veces brilla y a veces falla.

## 16. Entrenamiento final y evaluación en prueba (una sola vez)

Con la configuración elegida, entrenamos un modelo final con **todos** los datos de entrenamiento y validación, usando el número de épocas que salió de la validación cruzada. Recién ahora usamos el conjunto de prueba.

In [ ]:
modelo_final, _ = entrenar(X_todo, y_todo, dim=MEJOR_DIM, epocas=EPOCAS_FINALES, semilla=SEMILLA)
acc_te, f1_te, pred_te = evaluar(modelo_final, X_te, y_te)

vec_final, clf_final = entrenar_base(textos_todo, y_todo)
pred_base_te = clf_final.predict(vec_final.transform(test["nota_norm"].values))

print("=== RESULTADO EN PRUEBA (notas nunca vistas) ===")
print(f"Red neuronal (dim={MEJOR_DIM}): exactitud {acc_te:.3f} | F1 macro {f1_te:.3f}")
print(f"Modelo base TF-IDF:      exactitud {accuracy_score(y_te, pred_base_te):.3f} | "
      f"F1 macro {f1_score(y_te, pred_base_te, average='macro'):.3f}\n")
print(classification_report(y_te, pred_te, target_names=CLASES, zero_division=0))

In [ ]:
mc = confusion_matrix(y_te, pred_te, labels=range(len(CLASES)))
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(mc, cmap="Blues")
ax.set_xticks(range(len(CLASES)), CLASES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASES)), CLASES)
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        ax.text(j, i, mc[i, j], ha="center", va="center",
                color="white" if mc[i, j] > mc.max() / 2 else "black")
ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusión (prueba)")
plt.tight_layout(); plt.show()

In [ ]:
errores = test.assign(real=test["categoria"].values,
                      prediccion=[CLASES[p] for p in pred_te])
errores = errores[errores["real"] != errores["prediccion"]]
print(f"Notas mal clasificadas: {len(errores)} de {len(test)}")
errores[["nota", "real", "prediccion"]].head(10)

**Análisis de errores:** la mayoría de los errores son notas que **mezclan dos temas** ("dormí pésimo, harta pega en el trabajo"). La etiqueta solo dice el tema de la primera frase, así que el modelo no tiene forma de saber cuál es "el principal". No es un error del modelo, sino de cómo se etiquetaron los datos. La mejora natural es permitir **varias etiquetas por nota** (clasificación multietiqueta).

## 17. Usar el modelo y reentrenarlo con notas nuevas

En la vida real llegan notas nuevas todos los días. El ciclo es:
1. El modelo **predice** la categoría de cada nota nueva.
2. Si el usuario **corrige** la predicción, esa nota pasa a ser un dato de entrenamiento.
3. Cada cierto tiempo se **reentrena** con todo lo acumulado.

**Limitación:** el vocabulario quedó fijo en la Parte 1, así que las palabras nuevas se convierten en `<unk>` hasta que se reconstruya el vocabulario.

In [ ]:
def predecir(texto, modelo=None):
    modelo = modelo or modelo_final
    tokens = tokenizar_palabras(normalizar(anonimizar(texto)))
    modelo.eval()
    with torch.no_grad():
        probs = torch.softmax(modelo(torch.tensor([a_ids(tokens)])), dim=1)[0].numpy()
    orden = probs.argsort()[::-1]
    print(f"{texto!r}")
    print("   " + " | ".join(f"{CLASES[i]} {probs[i]:.0%}" for i in orden[:3]))
    return CLASES[orden[0]]

predecir("no pude dormir nada, me quedé pegado en el celular")
predecir("fui a trotar al parque 💪")
predecir("tomé como 5 cafés y almorcé puras papas fritas")
_ = predecir("mucho estrés con el certamen de cálculo 😩")

In [ ]:
X_acumulado, y_acumulado = X_todo.copy(), y_todo.copy()

def reentrenar_con(notas_nuevas):
    # notas_nuevas: lista de (texto, categoria_correcta). Reentrena desde cero con todo lo acumulado.
    global X_acumulado, y_acumulado, modelo_final
    X_nuevo = np.array([a_ids(tokenizar_palabras(normalizar(anonimizar(t)))) for t, _ in notas_nuevas])
    y_nuevo = np.array([clase_a_id[c] for _, c in notas_nuevas])
    X_acumulado = np.vstack([X_acumulado, X_nuevo])
    y_acumulado = np.concatenate([y_acumulado, y_nuevo])

    modelo_final, _ = entrenar(X_acumulado, y_acumulado, dim=MEJOR_DIM, epocas=EPOCAS_FINALES, semilla=SEMILLA)
    acc, f1, _ = evaluar(modelo_final, X_te, y_te)
    print(f"Reentrenado con {len(X_acumulado)} notas (+{len(notas_nuevas)} nuevas). "
          f"Prueba: exactitud {acc:.3f} | F1 macro {f1:.3f}")

reentrenar_con([
    ("no pude dormir pensando en la prueba", "sueno"),
    ("me quedé pegado en el celular hasta las 4", "sueno"),
    ("hice una caminata larga con mi perro", "ejercicio"),
])
_ = predecir("no pude dormir nada, me quedé pegado en el celular")

> **Ojo:** aquí miramos el conjunto de prueba después de reentrenar solo para mostrar el ciclo. En un proyecto real, cada versión nueva del modelo se evalúa con un conjunto de prueba **nuevo**, porque si miras la prueba muchas veces terminas ajustando el modelo a ella sin darte cuenta.

In [ ]:
torch.save({"estado": modelo_final.state_dict(), "dim": MEJOR_DIM, "clases": CLASES},
           "modelo_habitos.pt")
print("Modelo guardado en modelo_habitos.pt")

## Conclusiones de la Parte 2

1. **Épocas:** el modelo aprende pasando muchas veces por los datos. La parada temprana evita que memorice.
2. **Repeticiones:** un solo entrenamiento no basta. La validación cruzada (10 entrenamientos por modelo) muestra el promedio **y** la variación, y permite elegir la configuración con evidencia.
3. **Modelo base primero:** comparar con TF-IDF + regresión logística dice si la red neuronal realmente aporta.
4. **El conjunto de prueba se usa una vez.** Todas las decisiones se tomaron con validación.
5. **El límite está en los datos, no en el modelo:** los errores vienen sobre todo de notas con dos temas y una sola etiqueta.